# BONUS · B3 · Delta Lake: Advanced Features

> **This module is optional.** It extends the Delta Lake foundations from `day1_02_bi_sql_analytics_architecture`. You can work through it independently if time allows. It builds disposable demo tables (prefixed `bonus_b3_`) and cleans up after itself.

## Learning Objectives

By the end of this notebook you can:

- inspect Delta table metadata using `DESCRIBE DETAIL` and `DESCRIBE HISTORY`,
- explain what the Delta transaction log (`_delta_log`) records for each operation,
- query previous versions of a table using Time Travel (`VERSION AS OF`),
- restore a table to a previous version after an accidental change,
- apply an upsert (update-or-insert) using `MERGE INTO`,
- explain what `OPTIMIZE` and `VACUUM` do and when to use them,
- add a `CHECK` constraint to enforce data quality at the table level.

## Context

`day1_02_bi_sql_analytics_architecture` introduced Delta Lake fundamentals and the Gold layer. This bonus module goes deeper on the operational features that keep Gold tables reliable over time: auditing every change, recovering from mistakes, merging incremental updates, and managing table lifecycle.

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

In [ ]:
required = [f"{GOLD}.fact_sales", f"{SILVER}.customers"]
missing = [t for t in required if not spark.catalog.tableExists(t)]
if missing:
    raise Exception(f"Missing tables: {missing}. Run data/generate_training_dataset.ipynb first.")
print("[OK] Pre-check passed")

## 1. DESCRIBE DETAIL — Physical Table Metadata

`DESCRIBE DETAIL` returns storage-level information about a Delta table: file count, size, location, and table properties. It is the fastest way to answer questions like: "How many files does this table have?" and "Where are the data files stored?"

| Column | What it tells you |
|---|---|
| `format` | storage format (`delta`) |
| `numFiles` | number of active Parquet files (after OPTIMIZE / VACUUM) |
| `sizeInBytes` | total size of active data files |
| `location` | path in cloud storage (ADLS / S3 / GCS) |
| `partitionColumns` | partitioning columns, if any |

In [ ]:
detail = spark.sql(f"DESCRIBE DETAIL {GOLD}.fact_sales")
visible = [c for c in ["format", "numFiles", "sizeInBytes", "location", "partitionColumns"] if c in detail.columns]
display(detail.select(*visible))

## 2. DESCRIBE HISTORY — The Full Audit Log

Every Delta write operation (INSERT, UPDATE, DELETE, MERGE, CREATE OR REPLACE) is recorded in the `_delta_log` folder as a JSON file. `DESCRIBE HISTORY` surfaces that log as a queryable table.

| Column | What it tells you |
|---|---|
| `version` | sequential version number; increments on every write |
| `timestamp` | when the operation ran |
| `operation` | what happened: `WRITE`, `MERGE`, `UPDATE`, `DELETE`, `RESTORE` |
| `operationMetrics` | rows inserted / updated / deleted |
| `userName` | who triggered the operation |

> **BI relevance:** `DESCRIBE HISTORY` lets you answer "when was this Gold table last refreshed?" and "how many rows changed in the last update?" without any additional logging infrastructure.

In [ ]:
history = spark.sql(f"DESCRIBE HISTORY {GOLD}.fact_sales LIMIT 10")
visible = [c for c in ["version", "timestamp", "operation", "operationMetrics"] if c in history.columns]
display(history.select(*visible))

## 3. Time Travel — Querying Previous Versions

Because Delta never overwrites files in-place (it adds new files and logically removes old ones), every historical version of the table is still queryable — up to the point where `VACUUM` removes old files.

```sql
-- Query by version number
SELECT * FROM gold.fact_sales VERSION AS OF 0

-- Query by timestamp
SELECT * FROM gold.fact_sales TIMESTAMP AS OF '2025-06-01 00:00:00'
```

**Common use cases:**
- "What did this KPI look like before yesterday's refresh?"
- "Was this customer in the Gold table before the last batch?"
- "Prove that the number changed between version 3 and version 5."

> **Time Travel is not a backup.** `VACUUM` removes old files permanently. After a VACUUM run, versions older than the retention window are gone.

In [ ]:
# Create a disposable Time Travel demo table
DEMO_TT = f"{GOLD}.bonus_b3_time_travel"

spark.sql(f"DROP TABLE IF EXISTS {DEMO_TT}")
spark.sql(f"""
    CREATE TABLE {DEMO_TT} (
        order_date   DATE,
        revenue      DOUBLE,
        order_count  BIGINT
    ) USING DELTA
""")

# Version 0 = CREATE TABLE (no data)
spark.sql(f"""
    INSERT INTO {DEMO_TT}
    SELECT order_date,
           SUM(line_revenue)     AS revenue,
           COUNT(DISTINCT order_id) AS order_count
    FROM {GOLD}.fact_sales
    WHERE status = 'Completed'
      AND order_date BETWEEN '2024-01-01' AND '2024-01-07'
    GROUP BY order_date
""")
print("Version 1: initial 7-day snapshot inserted")

spark.sql(f"UPDATE {DEMO_TT} SET revenue = revenue * 1.1 WHERE order_date = '2024-01-01'")
print("Version 2: January 1st revenue adjusted +10%")

spark.sql(f"DELETE FROM {DEMO_TT} WHERE order_date = '2024-01-07'")
print("Version 3: January 7th row deleted")

display(spark.sql(f"DESCRIBE HISTORY {DEMO_TT}").select("version", "timestamp", "operation"))

In [ ]:
# Compare current state vs Version 1 (before any updates or deletes)
print("Current state (after UPDATE + DELETE):")
display(spark.sql(f"SELECT * FROM {DEMO_TT} ORDER BY order_date"))

print("\nVersion 1 (original 7-day snapshot, before changes):")
display(spark.sql(f"SELECT * FROM {DEMO_TT} VERSION AS OF 1 ORDER BY order_date"))

## 4. RESTORE — Recovering from an Accidental Change

`RESTORE TABLE ... TO VERSION AS OF n` rolls the table back to a previous version. Crucially, it does not rewrite history — it creates a new version that makes the table look like it did at version `n`. The audit log is preserved.

This is the Delta recovery path for:
- accidental `DELETE FROM table` (without a `WHERE` clause)
- a bad `UPDATE` that corrupted values
- a `CREATE OR REPLACE` that overwrote the wrong table

In [ ]:
# Simulate disaster: delete all rows
spark.sql(f"DELETE FROM {DEMO_TT}")
print("DISASTER: all rows deleted")
print("Row count:", spark.table(DEMO_TT).count())

# Restore to version 1 (original snapshot)
spark.sql(f"RESTORE TABLE {DEMO_TT} TO VERSION AS OF 1")
print("\nRESTORED to Version 1")
print("Row count:", spark.table(DEMO_TT).count())
display(spark.table(DEMO_TT).orderBy("order_date"))

# RESTORE adds a new version — the audit trail is intact
print("\nHistory after RESTORE (includes the disaster and the recovery):")
display(spark.sql(f"DESCRIBE HISTORY {DEMO_TT}").select("version", "timestamp", "operation"))

## 5. MERGE INTO — Incremental Upserts

`MERGE INTO` is the workhorse of incremental Gold pipelines. In a single atomic operation it:

- **updates** existing rows when a match is found (customer changed region),
- **inserts** new rows when no match exists (new customer),
- optionally **deletes** target rows with no matching source (customer removed).

```sql
MERGE INTO target AS t
USING source AS s ON t.key = s.key
WHEN MATCHED THEN UPDATE SET t.col = s.col
WHEN NOT MATCHED THEN INSERT (col1, col2) VALUES (s.col1, s.col2)
```

For BI developers, MERGE matters because it explains why a Gold refresh can change some rows without replacing the whole table — enabling **incremental** pipelines that run in seconds rather than minutes.

In [ ]:
from pyspark.sql import functions as F

DEMO_MERGE = f"{GOLD}.bonus_b3_merge_demo"

# Base table: top 5 customers by revenue
spark.sql(f"""
    CREATE OR REPLACE TABLE {DEMO_MERGE}
    COMMENT 'Bonus B3 demo: MERGE INTO target table'
    AS
    SELECT
        customer_id,
        SUM(CASE WHEN status = 'Completed' THEN line_revenue ELSE 0 END) AS lifetime_revenue,
        MAX(order_date)    AS last_order_date,
        'Active'           AS customer_status
    FROM {GOLD}.fact_sales
    GROUP BY customer_id
    ORDER BY lifetime_revenue DESC
    LIMIT 10
""")
print("Base table (10 customers):")
display(spark.table(DEMO_MERGE).orderBy("lifetime_revenue", ascending=False))

In [ ]:
# Incoming update batch: 3 existing customers with changed status + 1 brand new customer
# Pick the top 3 customer_ids dynamically from the base table
top3 = [r["customer_id"] for r in spark.table(DEMO_MERGE).orderBy("lifetime_revenue", ascending=False).limit(3).collect()]
new_customer_id = 9999999  # guaranteed not to exist in the base table

batch_rows = [(cid, 99999.99, "2025-12-31", "Premium") for cid in top3]
batch_rows.append((new_customer_id, 1234.56, "2025-06-15", "New"))

from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType
batch_schema = StructType([
    StructField("customer_id",       LongType(),   False),
    StructField("lifetime_revenue",  DoubleType(), True),
    StructField("last_order_date",   StringType(), True),
    StructField("customer_status",   StringType(), True),
])
batch_df = spark.createDataFrame(batch_rows, batch_schema)
batch_df.createOrReplaceTempView("bonus_b3_update_batch")

print("Incoming update batch:")
display(batch_df)

spark.sql(f"""
    MERGE INTO {DEMO_MERGE} AS target
    USING bonus_b3_update_batch AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN
        UPDATE SET
            lifetime_revenue = source.lifetime_revenue,
            last_order_date  = CAST(source.last_order_date AS DATE),
            customer_status  = source.customer_status
    WHEN NOT MATCHED THEN
        INSERT (customer_id, lifetime_revenue, last_order_date, customer_status)
        VALUES (source.customer_id, source.lifetime_revenue, CAST(source.last_order_date AS DATE), source.customer_status)
""")

print("\nTable after MERGE (3 rows updated, 1 inserted):")
display(spark.table(DEMO_MERGE).orderBy("lifetime_revenue", ascending=False))

## 6. Table Constraints — Enforcing Data Quality at Write Time

Delta supports `CHECK` constraints that reject rows violating a condition at the moment of write. Unlike Silver-layer filters (which silently drop bad rows), table constraints **fail the write** and surface the problem immediately.

```sql
ALTER TABLE gold.bonus_b3_merge_demo
ADD CONSTRAINT positive_revenue CHECK (lifetime_revenue >= 0)
```

Constraints are best for stable, table-level rules that should **never** be violated — for example, a revenue column that must be non-negative, or a status column limited to known values.

| Approach | When bad data arrives | Best for |
|---|---|---|
| Silver `WHERE` filter | row is silently dropped | quality gates in pipelines |
| Delta `CHECK` constraint | write is rejected with an error | table-level rules that should never fail |
| DQX / Lakehouse Monitoring | tracked, quarantined, or alerted | operational quality tracking at scale |

In [ ]:
try:
    spark.sql(f"""
        ALTER TABLE {DEMO_MERGE}
        ADD CONSTRAINT positive_revenue CHECK (lifetime_revenue >= 0)
    """)
    print("[OK] Constraint 'positive_revenue' added")
except Exception as e:
    print(f"[INFO] Constraint may already exist: {e}")

# Attempt to insert a row with negative revenue — constraint rejects it
try:
    spark.sql(f"""
        INSERT INTO {DEMO_MERGE} VALUES (8888888, -500.00, '2025-01-01', 'Invalid')
    """)
    print("[WARN] Invalid row was inserted — check constraint support on this runtime")
except Exception as e:
    print(f"[OK] Constraint rejected the invalid write:\n{type(e).__name__}: {str(e)[:200]}")

# Valid insert passes
spark.sql(f"INSERT INTO {DEMO_MERGE} VALUES (7777777, 250.00, '2025-03-01', 'Regular')")
print("\n[OK] Valid row inserted")
print("Row count:", spark.table(DEMO_MERGE).count())

## 7. OPTIMIZE and VACUUM — Table Lifecycle

### OPTIMIZE

Many small writes (streaming, frequent MERGE operations) produce many small Parquet files. Small files hurt query performance because each file requires a separate read operation.

`OPTIMIZE` compacts small files into larger ones. For large tables with varying filter patterns, **Liquid Clustering** is the modern alternative — it adaptively organizes data without requiring fixed partitioning.

```sql
-- Compact small files
OPTIMIZE gold.fact_sales

-- Compact and co-locate rows by a common filter column (Z-Order — older approach)
OPTIMIZE gold.fact_sales ZORDER BY (order_date)
```

> **For BI developers:** you typically won't run OPTIMIZE yourself — it is scheduled by data engineers or triggered automatically by Predictive Optimization (Unity Catalog managed tables). Understanding it helps you interpret query profiles when a dashboard runs slowly.

### VACUUM

`VACUUM` removes Parquet files that are no longer referenced by any version in the transaction log. After VACUUM, Time Travel to versions older than the retention window becomes impossible.

```sql
-- Default retention: 7 days (168 hours)
VACUUM gold.fact_sales

-- Explicit retention (never go below 168h in production)
VACUUM gold.fact_sales RETAIN 168 HOURS
```

| Command | What it does | BI impact |
|---|---|---|
| `OPTIMIZE` | compact small files | faster scan on next query |
| `VACUUM` | remove old file versions | Time Travel window shrinks |
| Predictive Optimization | automates OPTIMIZE + VACUUM on UC managed tables | reduces need for manual maintenance |

In [ ]:
# Inspect the current file layout of fact_sales — shows effect of OPTIMIZE (or lack thereof)
detail = spark.sql(f"DESCRIBE DETAIL {GOLD}.fact_sales")
visible = [c for c in ["format", "numFiles", "sizeInBytes"] if c in detail.columns]
display(detail.select(*visible))

# Check Predictive Optimization setting
try:
    props = spark.sql(f"SHOW TBLPROPERTIES {GOLD}.fact_sales").collect()
    po_props = [r for r in props if "predictive" in r["key"].lower() or "optimization" in r["key"].lower()]
    if po_props:
        print("\nPredictive Optimization properties:")
        for r in po_props:
            print(f"  {r['key']} = {r['value']}")
    else:
        print("\nNo explicit Predictive Optimization properties set (inherited from catalog/schema defaults)")
except Exception as e:
    print(f"\n[INFO] Could not inspect table properties: {e}")

## Summary

| Feature | What it does | When to use |
|---|---|---|
| `DESCRIBE DETAIL` | file count, size, location | investigate table storage and file layout |
| `DESCRIBE HISTORY` | full operation audit log | check last refresh, row-level change metrics |
| `VERSION AS OF` | query table at previous version | audit, debugging, regression comparison |
| `RESTORE` | roll back to previous version | disaster recovery after accidental delete/overwrite |
| `MERGE INTO` | atomic upsert | incremental pipeline refreshes, CDC |
| `CHECK` constraint | reject bad writes at table level | stable, non-negotiable data quality rules |
| `OPTIMIZE` | compact small files | improve scan performance after many small writes |
| `VACUUM` | delete old file versions | reclaim storage; narrows Time Travel window |

**One-line rules:**
- `DESCRIBE HISTORY` before debugging a KPI change — it shows exactly what changed and when.
- `RESTORE` after an accidental delete — it is faster and safer than replaying the pipeline.
- `MERGE INTO` for incremental Gold — it is atomic and auditable.
- `VACUUM` shrinks the Time Travel window — use the default 7-day retention unless storage cost is critical.

## Cleanup

Drop the demo tables created by this notebook.

In [ ]:
bonus_tables = [
    f"{GOLD}.bonus_b3_time_travel",
    f"{GOLD}.bonus_b3_merge_demo",
]

for table in bonus_tables:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    print(f"[OK] Dropped: {table}")

print("\nAll bonus B3 demo tables removed.")